## Testing Script
Import all necessary Packages

In [17]:
import numpy as np
import scipy as sc
from linear_solvers import NumPyLinearSolver, HHL
from linear_solvers.matrices.tridiagonal_toeplitz import TridiagonalToeplitz
from qiskit.quantum_info import Statevector
from qiskit import Aer
from scipy.sparse import diags

np.set_printoptions(precision=3, suppress=True, linewidth=120)  # Suppress scientific notation, limit decimals, and set max width.


Now define the matrix (A) and vector (b). Ensure that the condition number is suffeciently low

In [18]:
matrix = np.array([[1, -1/3], [-1/3, 1]])
vector = np.array([1, 0])
e = 1 # error tolerance
print('matrix condition number:', np.linalg.cond(matrix))


matrix condition number: 1.9999999999999991


Create the instances for all of the different solvers.

In [19]:
# solve using numpy
numpy_solve = NumPyLinearSolver() 
numpy_solution = numpy_solve.solve(matrix, vector/np.linalg.norm(vector))

# solve using HHL with no simplification to matrix
naive_hhl = HHL(epsilon=e) # solve using HHL
naive_solution = naive_hhl.solve(matrix, vector)

# solve using HHL with tridiagonal toeplitz simplification
tridi_mat = TridiagonalToeplitz(1, 1, -1 / 3, trotter_steps=2) # convert matrix to tridiagonal toeplitz form
tridi_hhl = HHL(epsilon=e)
tridi_solution = tridi_hhl.solve(tridi_mat, vector)

# solve using Aer + exact reciprocal
aer_hhl = HHL(epsilon=e, reciprocal=True, quantum_instance=Aer.get_backend('aer_simulator'))
aer_solution = aer_hhl.solve(matrix, vector)

# solve using chebyshev
cheby_hhl = HHL(epsilon=e, reciprocal=False)
cheby_solution = cheby_hhl.solve(matrix, vector)

Inverse Chebyshev Number of intervals:  1
Inverse Chebyshev degree:  1
Breakpoints: [4, 7]
Constant: 0.25
Degree: 1


Now get the results. The get_solution_vector function converts the quantum output into component wise output

In [20]:
NUM_QUBITS = tridi_solution.state.num_qubits

def get_solution_vector(solution, N):
    """Extracts and normalizes simulated state vector
    from LinearSolverResult."""
    start = 2**(N-1)
    fin = 2**(N-1) + len(vector)
    # print('start:', start)
    # print('fin:', fin)
    solution_vector = Statevector(solution.state).data[start:fin].real
    
    norm = solution.euclidean_norm
    return norm * solution_vector / np.linalg.norm(solution_vector)

print('')
print('full classical solution vector:', numpy_solution.state)
print('full tridi solution vector:', get_solution_vector(tridi_solution, NUM_QUBITS))
print('full aer solution vector:', get_solution_vector(aer_solution, NUM_QUBITS))
print('full normal solution vector:', get_solution_vector(naive_solution, NUM_QUBITS))
print('full cheby solution vector:', get_solution_vector(cheby_solution, cheby_solution.state.num_qubits))

print('')
print('Error for Statevector', np.linalg.norm(naive_solution.euclidean_norm - numpy_solution.euclidean_norm))   # Error for Statevector method
print('Error for Aer simulation', np.linalg.norm(aer_solution.euclidean_norm - numpy_solution.euclidean_norm))  # Error for Aer simulation method
print('Error for Tridiagonal', np.linalg.norm(tridi_solution.euclidean_norm - numpy_solution.euclidean_norm))   # Error for Tridiagonal method
print('Error for Chebyshev', np.linalg.norm(cheby_solution.euclidean_norm - numpy_solution.euclidean_norm))   # Error for Chebyshev method
print('')

#Get error in the Euclidean norm of the solution vectors
print('Error for Statevector in solution vec', np.linalg.norm(get_solution_vector(naive_solution, NUM_QUBITS) - numpy_solution.state))   # Error for Statevector method
print('Error for Aer simulation in solution vec', np.linalg.norm(get_solution_vector(aer_solution, NUM_QUBITS) - numpy_solution.state))  # Error for Aer simulation method
print('Error for Tridiagonal in solution vec', np.linalg.norm(get_solution_vector(tridi_solution, NUM_QUBITS) - numpy_solution.state))   # Error for Tridiagonal method
print('Error for Chebyshev in solution vec', np.linalg.norm(get_solution_vector(cheby_solution,cheby_solution.state.num_qubits) - numpy_solution.state))   # Error for Chebyshev method


full classical solution vector: [1.125 0.375]
full tridi solution vector: [1.125 0.375]
full aer solution vector: [1.125 0.375]
full normal solution vector: [1.125 0.375]
full cheby solution vector: [ 0.795 -0.705]

Error for Statevector 3.1086244689504383e-15
Error for Aer simulation 3.1086244689504383e-15
Error for Tridiagonal 1.3322676295501878e-15
Error for Chebyshev 0.12328115704542864

Error for Statevector in solution vec 3.2652638748082423e-15
Error for Aer simulation in solution vec 3.2652638748082423e-15
Error for Tridiagonal in solution vec 1.3732700395566711e-15
Error for Chebyshev in solution vec 1.1292134375659886
